In [ ]:
print("=" * 72)
print("STEP 0 - Clone repo, install Python dependencies")
print("=" * 72)

!git clone -q https://github.com/uzairlol/ELICIT-fyp.git /kaggle/working/elicit
%cd /kaggle/working/elicit

!git log -1 --format="Cloned HEAD  : %h %s"
!git status -sb | head -1

print("Environment pre-check:")
!python --version
!free -h | head -2
!df -h /kaggle | tail -1

!pip install -qq -r requirements.txt
print("requirements.txt installed.")

In [ ]:
print("=" * 72)
print("STEP 1 - Install vLLM into an isolated venv and verify CUDA/GPU")
print("=" * 72)

!apt-get update -y -qq > /dev/null
!apt-get install -y -qq pciutils zstd > /dev/null
print("apt: pciutils + zstd installed.")

# The Kaggle base image is packed with preinstalled packages (cudf/cuml/RAPIDS,
# bigframes, protobuf 6.x, grpcio, ...) that a plain `pip install vllm` runs
# straight into broken dependency resolution ("pip's dependency resolver does
# not currently take into account..."). Installing vLLM into a fresh venv keeps
# its resolver clean and does not disturb the base env the simulation uses (the
# server is only talked to over HTTP).
print("Creating isolated venv for vLLM ...")
!rm -rf /kaggle/working/vllm-env
# `python -m venv` can fail on Kaggle because the base Python build has no
# ensurepip/venv installed; virtualenv bundles its own pip and is robust here.
!python -m pip install -q --upgrade pip virtualenv
!python -m virtualenv /kaggle/working/vllm-env
print("Isolated venv created at /kaggle/working/vllm-env.")

print("Installing vLLM into the venv (large CUDA torch wheel, may take a few minutes) ...")
!date -u +"vllm install start: %T"
# Strip PYTHONPATH/PYTHONHOME: Kaggle's base sitecustomize imports 'wrapt'
# (missing in the venv) and only spews startup noise (and can interfere with
# imports). The venv resolves its own packages via pyvenv.cfg instead.
!env -u PYTHONPATH -u PYTHONHOME /kaggle/working/vllm-env/bin/pip install -q --upgrade pip
!env -u PYTHONPATH -u PYTHONHOME /kaggle/working/vllm-env/bin/pip install vllm
!date -u +"vllm install done: %T"

print("vLLM installed. Venv versions + CUDA check:")
!env -u PYTHONPATH -u PYTHONHOME /kaggle/working/vllm-env/bin/python -c "import vllm, torch; print('vllm =', vllm.__version__, '| torch =', torch.__version__, '| cuda_available =', torch.cuda.is_available(), '| gpus =', torch.cuda.device_count())"

!nvidia-smi

In [ ]:
# Default path for the Kaggle T4 x2 session: GPTQ-Int4 (~9 GB, same ballpark
# as `qwen2.5:14b`) served with BOTH T4s via tensor parallelism = ~2x the
# throughput of the single-T4 GGUF option, and none of the GGUF loading edge
# cases (this vLLM build probes --model as a HF directory, which a raw .gguf
# cannot satisfy). GPTQ runs on Turing (sm75) via non-Marlin kernels (Marlin
# is sm80+; vLLM falls back automatically).
#
# WHY THESE FLAGS (Qwen2.5-14B: 48 layers, 8 KV heads, head_dim 128):
#   KV bytes/token per GPU (TP2) = 192 KiB (24 layers each)
#   per-GPU budget = 0.90 * 16 GiB = 14.4 GiB; weights ~9.0 GB / 2 = ~4.5 GiB
#   leftover KV/GPU = ~9.4 GiB => ~49k tokens/kernel
#   --max-num-seqs 32 with typical ~400-1,800 token calls => ~7.2 GiB/GPU
#   worst case (all 32 @ 8192) exceeds the pool: vLLM preempts/recomputes
#   short prefills -- it never spills KV to CPU RAM (off by default today).

import json
import os
import subprocess
import time
import urllib.request

from huggingface_hub import snapshot_download

MODEL_REF = "Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4"
VLLM_LOG = "/kaggle/working/vllm.log"
VLLM_BIN = "/kaggle/working/vllm-env/bin/vllm"

print("=" * 72)
print("STEP 2 - Download weights, then start vLLM server in the background")
print("=" * 72)

# Wipe any stale model cache from a previous attempt (the gguf alone was ~9 GB).
!rm -rf /kaggle/working/hf_cache

print(f"Downloading {MODEL_REF} (single ~9 GB safetensors) ...")
downloaded_at = time.time()
try:
    model_dir = snapshot_download(
        repo_id=MODEL_REF,
        cache_dir="/kaggle/working/hf_cache",
    )
except Exception as exc:
    raise RuntimeError(f"Failed to download {MODEL_REF}: {exc}") from None
print(f"Downloaded in {time.time() - downloaded_at:.0f}s -> {model_dir}")

cmd = [
    VLLM_BIN,
    "serve",
    model_dir,
    "--served-model-name",
    "qwen2.5-14b",
    "--tensor-parallel-size",
    "2",
    "--max-model-len",
    "8192",
    "--max-num-seqs",
    "32",
    "--gpu-memory-utilization",
    "0.90",
]

print("Launching vLLM with:")
print("  " + " ".join(cmd))
with open(VLLM_LOG, "wb") as log_handle:
    vllm_proc = subprocess.Popen(
        cmd,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env={
            key: value
            for key, value in os.environ.items()
            if key not in ("PYTHONPATH", "PYTHONHOME")
        }
        | {
            "HF_HOME": "/kaggle/working/hf_cache",
            "HF_HUB_CACHE": "/kaggle/working/hf_cache",
            "PYTHONNOUSERSITE": "1",
        },
    )
print(f"vLLM PID: {vllm_proc.pid}   log: {VLLM_LOG}")

started_at = time.time()
print("Waiting for vLLM to serve /v1/models (model load can take 5-15 min) ...")
ready = False
for attempt in range(1, 1351):
    if vllm_proc.poll() is not None:
        print(f"vLLM process EXITED early (code {vllm_proc.returncode}).")
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass
    if attempt % 45 == 0:
        elapsed = time.time() - started_at
        tail = subprocess.run(
            ["tail", "-5", VLLM_LOG],
            capture_output=True,
            text=True,
        ).stdout.strip()
        print(f"  ...still loading after {elapsed:.0f}s ({attempt} polls); log tail:")
        if tail:
            print("  " + tail.replace("\n", "\n  "))
    time.sleep(2)

if ready:
    elapsed = time.time() - started_at
    print(f"vLLM is READY after {elapsed:.1f}s.")
    with urllib.request.urlopen("http://127.0.0.1:8000/v1/models") as response:
        models = json.loads(response.read())
    for model in models.get("data", []):
        print(f"  serving model: {model.get('id')}")
else:
    print("\nvLLM did not come up. Last 60 lines of the log:")
    tail = subprocess.run(
        ["tail", "-60", VLLM_LOG],
        capture_output=True,
        text=True,
    ).stdout.strip()
    print(tail)
    raise RuntimeError("vLLM failed to start; see the log dump above.")

In [ ]:
import json
import urllib.request

print("=" * 72)
print("STEP 3 - GPU + model smoke test")
print("=" * 72)

!nvidia-smi
print("VRAM free (MiB), via torch:")
!python -c "import torch; print([round(torch.cuda.mem_get_info(i)[0] / 1024**2) for i in range(torch.cuda.device_count())])"

print("vLLM log tail:")
!tail -8 /kaggle/working/vllm.log

# End-to-end smoke test: one tiny chat completion through the served engine.
try:
    body = json.dumps(
        {
            "model": "qwen2.5-14b",
            "messages": [{"role": "user", "content": "Reply with exactly: READY"}],
            "max_tokens": 8,
            "temperature": 0.0,
        }
    ).encode("utf-8")
    request = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions",
        data=body,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        reply = json.loads(response.read())
    print(f"Smoke test reply: {reply['choices'][0]['message']['content'].strip()!r}")
except Exception as exc:
    print(f"Smoke test failed: {exc}. The server may still be starting or has crashed.")

In [ ]:
print("=" * 72)
print("STEP 4 - Resolve the runtime configuration the sweeps will use")
print("=" * 72)

import sys  # noqa: E402  (import must run after the sys.path.append below)

sys.path.append("/kaggle/working/elicit/src")
%cd /kaggle/working/elicit/src

from core import parameters  # noqa: E402
from llm import create_llm_client  # noqa: E402

client = create_llm_client()
print("Resolved runtime configuration:")
print(f"  LLM_BACKEND         = {parameters.LLM_BACKEND}")
print(f"  LLM_MODEL           = {parameters.LLM_MODEL}")
print(f"  LLM_BASE_URL        = {parameters.LLM_BASE_URL}")
print(f"  client class        = {type(client).__name__}")
print(f"  LLM_MAX_CONCURRENCY = {parameters.LLM_MAX_CONCURRENCY}")
print(f"  TOM_MAX_CONCURRENCY = {parameters.TOM_MAX_CONCURRENCY}")
print(f"  client max_inflight = {client.max_concurrency}")
print(f"  BELIEF_TRACKING     = {getattr(parameters, 'BELIEF_TRACKING_ENABLED', '<unset>')}")
print(f"  SUBSIDY_ENABLED     = {getattr(parameters, 'SUBSIDY_ENABLED', '<unset>')}")

# The baselines keep the institutional machinery switched OFF, but belief-state
# updates and subsidies are NOT gated by any sweep flag, so the persona baselines
# still issue a real LLM call per agent per round. The vLLM server is required.
ldf_counts = getattr(parameters, "LDF_AGENT_GROUP_COUNTS", {}) or {}
print(f"  LDF group counts    = {ldf_counts}  -> {sum(ldf_counts.values())} agents")
print(f"  abstract group counts = {getattr(parameters, 'AGENT_GROUP_COUNTS', {})}  -> "
      f"{sum((getattr(parameters, 'AGENT_GROUP_COUNTS', {}) or {}).values())} agents")

In [ ]:
print("=" * 72)
print("STEP 5 - Define the baseline grid")
print("=" * 72)

# ----------------------------------------------------------------------------
# WHAT IS BEING REPRODUCED
# ----------------------------------------------------------------------------
# The committed baseline artifacts (results/simulation_llama3.1_8b_Baseline_*)
# and analysis_baseline_studies.tex cover EIGHT abstract-scenario runs on
# llama3.1:8b:
#   Runs 1-4  Control / Reputation / Voting / Full  (institutional ablations)
#   Runs 5-6  Baseline_Random / Baseline_Greedy     (pure behavioural personas)
#   Runs 7-8  Full_Mixed_LLM5_Random{2,0}_Greedy{0,2} (mixed populations, 30 rounds)
# This notebook re-runs that whole suite on qwen2.5-14b so the baselines are
# model-matched to the main LDF runs already in results/, and then extends it
# with the same persona baselines under the climate/LDF regime.
#
# IMPORTANT: `Random` and `Greedy` are NOT hardcoded heuristics. They are LLM
# agents with behavioural-persona prompt blocks (core/personas.py, wired up in
# main.py::build_llm_agent), so they still hit the served model every round.
#
# WHY A NOTEBOOK DRIVER INSTEAD OF run_experiments.py FLAGS: the baseline loop
# is nested INSIDE the ablation gate (run_experiments.py: `if include_ablations
# and not full_only:`), so no stock flag combination isolates the baselines --
# `--skip-ablations` drops them too and `--skip-mixed` drags along the four
# ablations. Driving run_simulation() directly gives exact control over the
# batch composition, the scenario and the agent count, with no repo edits.
# ----------------------------------------------------------------------------

MODEL_NAME = "qwen2.5-14b"  # must equal the vLLM --served-model-name from STEP 2
SEEDS = [1]  # ABLATION_SEEDS: the baseline protocol is single-seed by design

# Agent counts are dictated by the profile tables, not by us:
#   abstract -> AGENT_GROUP_COUNTS        = 3 developed + 4 developing = 7
#   ldf      -> LDF_AGENT_GROUP_COUNTS    = 12 developed + 14 developing = 26
# main.py force-adjusts NUM_AGENTS to the LDF total whenever SCENARIO == "ldf",
# and mixed populations raise ValueError unless the mix sums to NUM_AGENTS.
ABSTRACT_NUM_AGENTS = 7
LDF_NUM_AGENTS = 26

# Round horizons, taken from the sweep constants in run_experiments.py:
#   ABLATION_NUM_ROUNDS = 20 (ablations + pure baselines)
#   MAIN_NUM_ROUNDS     = 30 (mixed populations + the main LDF Full runs)
ABLATION_ROUNDS = 20
MIXED_ROUNDS = 30
# The LDF baseline is horizon-matched to the main LDF Full runs already in
# results/ (30 rounds) so the counterfactual is directly comparable. Set this
# to ABLATION_ROUNDS to instead contrast LDF against the abstract baselines at
# a fixed 20-round horizon.
LDF_BASELINE_ROUNDS = 30

def gov(tom=False, gossip=False, voting=False):
    """Governance flags, matching the Control/Reputation/Voting/Full ablations."""
    return dict(tom=tom, gossip=gossip, voting=voting)


FULL_GOV = gov(tom=True, gossip=True, voting=True)
TOM_GOSSIP_GOV = gov(tom=True, gossip=True)  # the "Reputation" ablation
VOTING_GOV = gov(voting=True)  # the "Voting" ablation
NO_GOV = gov()  # the pure-persona baselines

# ----------------------------------------------------------------------------
# STAGE A -- abstract boundary baselines (no shocks, no LDF, 7 agents)
# ----------------------------------------------------------------------------
STAGE_A_RUNS = [
    dict(batch="Control", agent_type="LLM", mix=None, rounds=ABLATION_ROUNDS, **NO_GOV),
    dict(batch="Reputation", agent_type="LLM", mix=None, rounds=ABLATION_ROUNDS, **TOM_GOSSIP_GOV),
    dict(batch="Voting", agent_type="LLM", mix=None, rounds=ABLATION_ROUNDS, **VOTING_GOV),
    dict(batch="Full", agent_type="LLM", mix=None, rounds=ABLATION_ROUNDS, **FULL_GOV),
    dict(batch="Baseline_Random", agent_type="Random", mix=None, rounds=ABLATION_ROUNDS, **NO_GOV),
    dict(batch="Baseline_Greedy", agent_type="Greedy", mix=None, rounds=ABLATION_ROUNDS, **NO_GOV),
    dict(batch="Full_Mixed_LLM5_Random2_Greedy0", agent_type="LLM",
         mix={"LLM": 5, "Random": 2, "Greedy": 0}, rounds=MIXED_ROUNDS, **FULL_GOV),
    dict(batch="Full_Mixed_LLM5_Random1_Greedy1", agent_type="LLM",
         mix={"LLM": 5, "Random": 1, "Greedy": 1}, rounds=MIXED_ROUNDS, **FULL_GOV),
    dict(batch="Full_Mixed_LLM5_Random0_Greedy2", agent_type="LLM",
         mix={"LLM": 5, "Random": 0, "Greedy": 2}, rounds=MIXED_ROUNDS, **FULL_GOV),
]

# ----------------------------------------------------------------------------
# STAGE B -- climate/LDF-matched baselines (shocks + LDF on, 26 agents)
# ----------------------------------------------------------------------------
# The pure-persona counterfactuals under the regime the thesis cares about:
# does the LDF still matter when nobody funds it? Governance stays OFF, which
# is what makes these "baselines" rather than ablations. The pure-LLM LDF
# reference point is the existing `Full_scnldf_sh1_ldf1_*_26agents_30rounds`
# sweep in results/ (seeds 1-9), so it is NOT re-run here: it would cost
# 26*25 = 650 ToM audits per round for a number that already exists.
STAGE_B_RUNS = [
    dict(batch="Baseline_Random", agent_type="Random", mix=None,
         rounds=LDF_BASELINE_ROUNDS, **NO_GOV),
    dict(batch="Baseline_Greedy", agent_type="Greedy", mix=None,
         rounds=LDF_BASELINE_ROUNDS, **NO_GOV),
]

# Mixed populations under LDF are possible but expensive and OFF by default.
# The abstract 5:2 LLM-to-minority ratio is projected onto 26 agents: 26*5/7 =
# 18.57 -> a 19-agent LLM block, leaving 7 minority slots. A *strictly* equal
# 1:1 split is impossible against a fixed 19-agent LLM block because 7 is odd,
# so the two runs below use the closest distinct splits (4:3 and 3:4). If an
# exact 1:1 is required instead, drop to 18 LLM + 4 + 4 and accept a 69% LLM
# share against 73% for the other two conditions.
# Cost per round for these is
#   26 (institution) + 26 (contribution) + 26 (punishment) + 26 (belief)
#   + 26*25 = 650 ToM audits  =  ~754 LLM calls
# i.e. 30 rounds is >22k calls for ONE run. Flip RUN_LDF_MIXED to True only in
# a session you are happy to dedicate to it.
STAGE_B_MIXED_RUNS = [
    dict(batch="Full_Mixed_LLM19_Random4_Greedy3", agent_type="LLM",
         mix={"LLM": 19, "Random": 4, "Greedy": 3}, rounds=LDF_BASELINE_ROUNDS, **FULL_GOV),
    dict(batch="Full_Mixed_LLM19_Random3_Greedy4", agent_type="LLM",
         mix={"LLM": 19, "Random": 3, "Greedy": 4}, rounds=LDF_BASELINE_ROUNDS, **FULL_GOV),
]
RUN_LDF_MIXED = False


def estimate_llm_calls(spec, num_agents, rounds):
    """Rough per-run LLM call budget: institution + contribution + punishment
    + belief-state update per agent per round, plus the O(N^2) ToM audit sweep
    and one democratic session per DEMOCRACY_INTERVAL rounds."""
    per_round = 4 * num_agents
    if spec.get("tom"):
        per_round += num_agents * (num_agents - 1)
    if spec.get("voting"):
        per_round += num_agents / max(1, parameters.DEMOCRACY_INTERVAL)
    return int(per_round * rounds)


def show_plan(title, runs, scenario, num_agents, shocks, ldf):
    total = 0
    print(f"\n{title}   (scenario={scenario}, agents={num_agents}, "
          f"shocks={shocks}, ldf={ldf}, seeds={SEEDS})")

    def pop_label(spec):
        mix = spec.get("mix")
        if mix is None:
            return "pure " + spec["agent_type"]
        return f"LLM{mix['LLM']}+Rnd{mix['Random']}+Grd{mix['Greedy']}"

    def gov_label(spec):
        on = [name for name, key in (("ToM", "tom"), ("Gossip", "gossip"),
                                     ("Voting", "voting")) if spec.get(key)]
        return "+".join(on) if on else "none"

    batch_w = max(len("batch"), *(len(s["batch"]) for s in runs))
    pop_w = max(len("population"), *(len(pop_label(s)) for s in runs))
    print("-" * (batch_w + pop_w + 30))
    print(f"{'batch':<{batch_w}}  {'population':<{pop_w}}  {'governance':<18}  {'rnds':>4}  {'~calls':>9}")
    print("-" * (batch_w + pop_w + 30))
    for spec in runs:
        calls = estimate_llm_calls(spec, num_agents, spec["rounds"]) * len(SEEDS)
        total += calls
        print(f"{spec['batch']:<{batch_w}}  {pop_label(spec):<{pop_w}}  "
              f"{gov_label(spec):<18}  {spec['rounds']:>4}  {calls:>9,}")
    print("-" * (batch_w + pop_w + 30))
    print(f"{'TOTAL ~LLM calls':<{batch_w + pop_w + 24}}{total:>9,}")
    return total


plan_a = show_plan("STAGE A -- abstract boundary baselines", STAGE_A_RUNS,
                   "abstract", ABSTRACT_NUM_AGENTS, False, False)
plan_b = show_plan("STAGE B -- climate/LDF baselines", STAGE_B_RUNS,
                   "ldf", LDF_NUM_AGENTS, True, True)
if RUN_LDF_MIXED:
    plan_b += show_plan("STAGE B (opt-in) -- mixed LDF populations", STAGE_B_MIXED_RUNS,
                        "ldf", LDF_NUM_AGENTS, True, True)
n_runs = (len(STAGE_A_RUNS) + len(STAGE_B_RUNS)) * len(SEEDS)
opt_in = f", plus {len(STAGE_B_MIXED_RUNS)} opt-in conditions" if RUN_LDF_MIXED else ""
if RUN_LDF_MIXED:
    n_runs += len(STAGE_B_MIXED_RUNS) * len(SEEDS)
print(f"\nGrand total ~{plan_a + plan_b:,} LLM calls across {n_runs} run(s): "
      f"{len(STAGE_A_RUNS) + len(STAGE_B_RUNS)} conditions x {len(SEEDS)} seed(s)"
      f"{opt_in}.")
print("STEP 7 and STEP 8 are deliberately separate cells: the sweep is ~17.8k LLM")
print("calls, so on Kaggle you can run Stage A, archive it (STEP 10), and only then")
print("spend the rest of the session on Stage B. Re-running a cell re-runs its runs.")

In [ ]:
print("=" * 72)
print("STEP 6 - Define the sweep driver (no runs happen in this cell)")
print("=" * 72)

import time  # noqa: E402
import traceback  # noqa: E402

import main as main_mod  # noqa: E402
import run_experiments as rx  # noqa: E402

main_mod.setup_logging()  # same file+stream logging the CLI gets

RUN_LOG = []


def run_grid(runs, stage, scenario, num_agents, shocks, ldf, seeds=SEEDS):
    """Drive run_experiments.run_simulation() over a list of batch specs.

    Every condition in `runs` is run at every seed in `seeds` -- the seed loop
    is the OUTER loop, so the default SEEDS = [1] means "seed 1 for all
    conditions", not "one condition".

    run_simulation() reads its scenario / agent-count / climate switches from
    the run_experiments MODULE globals (SCENARIO, NUM_AGENTS,
    ENABLE_CLIMATE_SHOCKS, ENABLE_LDF, MODEL), so those are set here rather than
    passed per call. Each spec then supplies the governance flags, the agent
    persona, the optional mixed population and the round horizon.
    """
    rx.MODEL = MODEL_NAME
    rx.SCENARIO = scenario
    rx.NUM_AGENTS = num_agents
    rx.ENABLE_CLIMATE_SHOCKS = shocks
    rx.ENABLE_LDF = ldf
    rx.VERBOSE = True
    rx.current_run = 0
    rx.TOTAL_RUNS = len(runs) * len(seeds)

    for seed in seeds:
        for spec in runs:
            mix = spec.get("mix")
            if mix is not None and sum(mix.values()) != num_agents:
                raise ValueError(
                    f"mixed population {mix} sums to {sum(mix.values())}, "
                    f"but scenario={scenario} uses NUM_AGENTS={num_agents}"
                )
            print("=" * 78)
            print(f"[{stage}] {spec['batch']} | seed={seed} | agents={num_agents} "
                  f"| rounds={spec['rounds']} | persona={spec['agent_type']} "
                  f"| tom={spec['tom']} gossip={spec['gossip']} voting={spec['voting']} "
                  f"| mix={mix} | shocks={shocks} ldf={ldf}")
            print("=" * 78)
            started = time.time()
            status = "ok"
            try:
                rx.run_simulation(
                    batch_name=spec["batch"],
                    seed=seed,
                    tom_enabled=spec["tom"],
                    gossip_enabled=spec["gossip"],
                    democracy_enabled=spec["voting"],
                    agent_type=spec["agent_type"],
                    mixed_agent_counts=mix,
                    num_rounds=spec["rounds"],
                )
            except Exception as exc:  # keep the sweep alive across failures
                status = f"FAILED: {type(exc).__name__}: {exc}"
                traceback.print_exc()
            elapsed = (time.time() - started) / 60.0
            print(f"[{stage}] {spec['batch']} seed={seed} -> {status} "
                  f"({elapsed:.1f} min, run {rx.current_run}/{rx.TOTAL_RUNS})")
            RUN_LOG.append({
                "stage": stage,
                "batch": spec["batch"],
                "seed": seed,
                "scenario": scenario,
                "num_agents": num_agents,
                "rounds": spec["rounds"],
                "agent_type": spec["agent_type"],
                "mix": mix,
                "tom": spec["tom"],
                "gossip": spec["gossip"],
                "voting": spec["voting"],
                "shocks": shocks,
                "ldf": ldf,
                "minutes": round(elapsed, 1),
                "status": status,
            })


def show_run_log():
    print("\n" + "=" * 78)
    print("SWEEP SUMMARY (cumulative across every run_grid call in this session)")
    print("=" * 78)
    for row in RUN_LOG:
        print(f"[{row['stage']:<7}] {row['batch']:<34} seed={row['seed']} "
              f"{row['rounds']:>3}r  {row['minutes']:>6} min  {row['status']}")
    failed = [r for r in RUN_LOG if r["status"] != "ok"]
    print(f"\n{len(RUN_LOG) - len(failed)}/{len(RUN_LOG)} runs completed.")

In [ ]:
print("=" * 72)
print("STEP 7 - Run Stage A: the abstract boundary baselines")
print("=" * 72)

run_grid(STAGE_A_RUNS, "A", "abstract", ABSTRACT_NUM_AGENTS, False, False)

show_run_log()

In [ ]:
print("=" * 72)
print("STEP 8 - Run Stage B: the climate/LDF-matched baselines")
print("=" * 72)

run_grid(STAGE_B_RUNS, "B", "ldf", LDF_NUM_AGENTS, True, True)
if RUN_LDF_MIXED:
    run_grid(STAGE_B_MIXED_RUNS, "B-mixed", "ldf", LDF_NUM_AGENTS, True, True)

show_run_log()

In [ ]:
print("=" * 72)
print("STEP 9 - Summarise the baseline runs")
print("=" * 72)

import csv  # noqa: E402
import glob  # noqa: E402
import json as json_mod  # noqa: E402
import re  # noqa: E402
from pathlib import Path  # noqa: E402

RESULTS_DIR = "/kaggle/working/elicit/results"
SUMMARY_CSV = "/kaggle/working/baseline_summary.csv"

# core/environment.py::save_results writes
#   simulation_{model}_{batch}_scn{scn}_sh{sh}_ldf{ldf}_seed{seed}_{n}agents_{r}rounds_{ts}.json
# where {scn} is the bare scenario key (abstract / ldf). Note the literal "scn"
# is part of the filename template, not the scenario value.
# Note: src/analysis/export_ablation_{metrics,plots}.py parse the condition with
# r"8b_(.+?)_scn", which hard-codes the old llama3.1_8b model tag and therefore
# silently yields "unknown" for every qwen2.5-14b file (including the existing
# Full_scnldf runs). That is why the parser below is model-agnostic.
TAG_RE = re.compile(
    r"_scn(?P<scn>[a-z]+)_sh(?P<sh>[01])_ldf(?P<ldf>[01])_seed(?P<seed>\d+)"
    r"_(?P<agents>\d+)agents_(?P<rounds>\d+)rounds_(?P<ts>\d{8}_\d{6})\.json$"
)
MODEL_PREFIX = f"simulation_{MODEL_NAME}_"


def parse_results_file(path):
    name = Path(path).name
    tag = TAG_RE.search(name)
    if not tag or not name.startswith(MODEL_PREFIX):
        return None
    return {
        "file": name,
        "condition": name[len(MODEL_PREFIX):tag.start()],
        "scenario": tag.group("scn"),
        "shocks": int(tag.group("sh")),
        "ldf": int(tag.group("ldf")),
        "seed": int(tag.group("seed")),
        "agents": int(tag.group("agents")),
        "rounds": int(tag.group("rounds")),
    }


def summarise(path, meta):
    with open(path, encoding="utf-8") as fh:
        rounds = json_mod.load(fh)
    if not rounds:
        return None

    last = rounds[-1]
    last_agents = last.get("agents", {})

    contributions, parser_failures = [], 0
    institution, switches = {}, 0
    ldf_in, ldf_out, shocks = 0.0, 0.0, 0
    for rnd in rounds:
        if rnd.get("shock_occurred"):
            shocks += 1
        ldf_in += float(rnd.get("ldf_contributions_total") or 0.0)
        ldf_out += float(rnd.get("ldf_payouts_total") or 0.0)
        for aid, row in (rnd.get("agents") or {}).items():
            contributions.append(float(row.get("contribution") or 0.0))
            parser_failures += int(row.get("parsing_failures") or 0)
            choice = row.get("institution_choice")
            if aid in institution and institution[aid] != choice:
                switches += 1
            institution[aid] = choice

    def mean(values):
        return sum(values) / len(values) if values else 0.0

    dev = [r["cumulative_payoff"] for r in last_agents.values()
           if r.get("agent_group") == "developed"]
    devg = [r["cumulative_payoff"] for r in last_agents.values()
            if r.get("agent_group") == "developing"]
    strategies = sorted({r.get("strategy", "?") for r in last_agents.values()})
    sfi = len(last.get("sfi_members") or [])
    si = len(last.get("si_members") or [])
    shock_tag = "S" if meta["shocks"] else "-"
    ldf_tag = "L" if meta["ldf"] else "-"

    return {
        "run_id": f"{meta['condition']}[{meta['scenario']}{shock_tag}{ldf_tag}]s{meta['seed']}",
        "condition": meta["condition"],
        "scenario": meta["scenario"],
        "shocks_on": meta["shocks"],
        "ldf_on": meta["ldf"],
        "seed": meta["seed"],
        "agents": meta["agents"],
        "rounds_recorded": len(rounds),
        "mean_contribution": round(mean(contributions), 2),
        "institution_switches": switches,
        "final_SI": si,
        "final_SFI": sfi,
        "mean_final_payoff": round(mean([r.get("cumulative_payoff", 0.0)
                                        for r in last_agents.values()]), 1),
        "developed_final": round(mean(dev), 1),
        "developing_final": round(mean(devg), 1),
        "dev_gap": round(mean(dev) - mean(devg), 1),
        "gini_final": round(float(last.get("gini_wealth") or 0.0), 3),
        "cooperation_final": round(float(last.get("cooperation_rate") or 0.0), 3),
        "tom_used": any(r.get("tom_scores") for rnd in rounds
                        for r in (rnd.get("agents") or {}).values()),
        "democracy_rounds": sum(1 for rnd in rounds if rnd.get("constitutional_change")),
        "ldf_contributed": round(ldf_in, 1),
        "ldf_paid_out": round(ldf_out, 1),
        "shock_rounds": shocks,
        "parser_failures": parser_failures,
        "strategies": "|".join(strategies),
    }


rows = []
for path in sorted(glob.glob(f"{RESULTS_DIR}/simulation_{MODEL_NAME}_*.json")):
    meta = parse_results_file(path)
    if not meta:
        continue
    summary = summarise(path, meta)
    if summary:
        rows.append(summary)

rows.sort(key=lambda r: (r["scenario"] != "abstract", r["condition"], r["seed"]))
if not rows:
    print("No qwen2.5-14b baseline result files found yet -- run STEP 7/8 first.")
else:
    print(f"{len(rows)} run(s) found in {RESULTS_DIR}\n")

    def show_table(title, columns):
        print(f"\n{title}")
        widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in columns}
        print("  ".join(c.ljust(widths[c]) for c in columns))
        print("-+-".join("-" * widths[c] for c in columns))
        for r in rows:
            print("  ".join(str(r[c]).ljust(widths[c]) for c in columns))

    show_table("CONFIGURATION", ["run_id", "scenario", "shocks_on", "ldf_on", "seed",
                                 "agents", "rounds_recorded", "tom_used",
                                 "democracy_rounds", "strategies"])
    show_table("OUTCOMES", ["run_id", "mean_contribution", "institution_switches",
                            "final_SI", "final_SFI", "mean_final_payoff",
                            "developed_final", "developing_final", "dev_gap",
                            "gini_final", "cooperation_final", "shock_rounds",
                            "ldf_contributed", "ldf_paid_out", "parser_failures"])

    csv_columns = list(rows[0].keys())
    with open(SUMMARY_CSV, "w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=csv_columns, extrasaction="ignore")
        writer.writeheader()
        for r in rows:
            writer.writerow(r)
    print(f"\nCSV written: {SUMMARY_CSV}")

    print("\nGroup contrast (the 'invariance of structural inequality' claim):")
    for r in rows:
        print(f"  {r['run_id']:<44} developed {r['developed_final']:>14,.1f} | "
              f"developing {r['developing_final']:>14,.1f} | gap {r['dev_gap']:>+14,.1f}")

In [ ]:
print("=" * 72)
print("STEP 10 - Archive results")
print("=" * 72)

import glob  # noqa: E402
import os  # noqa: E402
import subprocess  # noqa: E402

REPO_ROOT = "/kaggle/working/elicit"
RESULTS_DIR = os.path.join(REPO_ROOT, "results")
ARCHIVE = "/kaggle/working/baseline_results.tar.gz"
SUMMARY_CSV = "/kaggle/working/baseline_summary.csv"  # written by STEP 9

produced = sorted(
    os.path.basename(p)
    for p in glob.glob(os.path.join(RESULTS_DIR, f"simulation_{MODEL_NAME}_*.json"))
)
print(f"{len(produced)} qwen2.5-14b result file(s) in {RESULTS_DIR}:\n")
for name in produced:
    size_mb = os.path.getsize(os.path.join(RESULTS_DIR, name)) / 1024**2
    print(f"  {size_mb:6.2f} MB  {name}")

if os.path.exists(SUMMARY_CSV):
    print(f"\n  {os.path.getsize(SUMMARY_CSV):6.2f} MB  {SUMMARY_CSV}")

# results/ only. The main-run notebook also tars data/, but that directory does
# not exist in the clone, which makes tar exit non-zero.
print("\nArchiving ...")
if os.path.exists(ARCHIVE):
    os.remove(ARCHIVE)
subprocess.run(["tar", "-czf", ARCHIVE, "-C", REPO_ROOT, "results"], check=True)

listing = subprocess.run(
    ["tar", "-tzf", ARCHIVE], capture_output=True, text=True, check=True
).stdout.splitlines()
print(f"Archive contains {len(listing)} entries, size "
      f"{os.path.getsize(ARCHIVE) / 1024**2:.2f} MB")
for entry in listing[:40]:
    print(f"  {entry}")